# Machine Learning Exercise 2
by Thanh Tran, Gero Brunke, Robin-Marcel Hanne

## Task 3

In [1]:
import numpy as np
from sklearn import datasets

####################################

class ReLULayer(object):
    def forward(self, input):
        # remember the input for later backpropagation
        self.input = input
        # return the ReLU of the input
        # pass positive values unchanged, set negatives to zero
        relu = np.maximum(0, input) # your code here
        return relu

    def backward(self, upstream_gradient):
        # compute the derivative of ReLU from upstream_gradient and the stored input

        # Derivative of ReLU is 1 where input > 0, else 0.
        # Multiply upstream gradient element-wise by this mask (chain rule).
        downstream_gradient = upstream_gradient * (self.input > 0) # your code here
        return downstream_gradient

    def update(self, learning_rate):
        pass # ReLU is parameter-free

####################################

class OutputLayer(object):
    def __init__(self, n_classes):
        self.n_classes = n_classes

    def forward(self, input):
        # remember the input for later backpropagation
        self.input = input
        # return the softmax of the input
        # subtract row-wise max before exp to prevent overflow
        shifted = input - input.max(axis=1, keepdims=True)
        exp_x = np.exp(shifted)
        softmax = exp_x / exp_x.sum(axis=1, keepdims=True) # your code here
        return softmax

    def backward(self, predicted_posteriors, true_labels):
        # return the loss derivative with respect to the stored inputs
        # (use cross-entropy loss and the chain rule for softmax,
        #  as derived in the lecture)
        batch_size = predicted_posteriors.shape[0]

        # Build one-hot matrix for true labels
        one_hot = np.zeros_like(predicted_posteriors)
        one_hot[np.arange(batch_size), true_labels] = 1.0

        # Combined derivative of cross-entropy loss through softmax (chain rule)
        # Divide by batch_size because the loss is a mean over samples, not a sum.
        downstream_gradient = (predicted_posteriors - one_hot) / batch_size
        return downstream_gradient # your code here

        return downstream_gradient

    def update(self, learning_rate):
        pass # softmax is parameter-free

####################################

class LinearLayer(object):
    def __init__(self, n_inputs, n_outputs):
        self.n_inputs  = n_inputs
        self.n_outputs = n_outputs
        # randomly initialize weights and intercepts
        # scale by sqrt(2/n_inputs)
        self.B = np.random.normal(0, np.sqrt(2.0 / n_inputs), (n_inputs, n_outputs)) # your code here
        self.b = np.random.normal(0, np.sqrt(2.0 / n_inputs), (n_outputs,)) # your code here

    def forward(self, input):
        # remember the input for later backpropagation
        self.input = input
        # compute the scalar product of input and weights
        # (these are the preactivations for the subsequent non-linear layer)

        # Affine transformation: preactivations = input @ B + b
        preactivations = input @ self.B + self.b # your code here
        return preactivations

    def backward(self, upstream_gradient):
        # compute the derivative of the weights from
        # upstream_gradient and the stored input

        # Gradient w.r.t. bias: sum upstream gradients over the batch (column-wise)
        self.grad_b = upstream_gradient.sum(axis=0) # your code here

        # Gradient w.r.t. weight matrix B:
        self.grad_B = self.input.T @ upstream_gradient # your code here

        # Downstream gradient passed to the preceding layer
        downstream_gradient = upstream_gradient @ self.B.T # your code here
        return downstream_gradient

    def update(self, learning_rate):
        # update the weights by batch gradient descent
        self.B = self.B - learning_rate * self.grad_B
        self.b = self.b - learning_rate * self.grad_b

####################################

class MLP(object):
    def __init__(self, n_features, layer_sizes):
        # constuct a multi-layer perceptron
        # with ReLU activation in the hidden layers and softmax output
        # (i.e. it predicts the posterior probability of a classification problem)
        #
        # n_features: number of inputs
        # len(layer_size): number of layers
        # layer_size[k]: number of neurons in layer k
        # (specifically: layer_sizes[-1] is the number of classes)
        self.n_layers = len(layer_sizes)
        self.layers   = []

        # create interior layers (linear + ReLU)
        n_in = n_features
        for n_out in layer_sizes[:-1]:
            self.layers.append(LinearLayer(n_in, n_out))
            self.layers.append(ReLULayer())
            n_in = n_out

        # create last linear layer + output layer
        n_out = layer_sizes[-1]
        self.layers.append(LinearLayer(n_in, n_out))
        self.layers.append(OutputLayer(n_out))

    def forward(self, X):
        # X is a mini-batch of instances
        batch_size = X.shape[0]
        # flatten the other dimensions of X (in case instances are images)
        X = X.reshape(batch_size, -1)

        # compute the forward pass
        # (implicitly stores internal activations for later backpropagation)
        result = X
        for layer in self.layers:
            result = layer.forward(result)
        return result

    def backward(self, predicted_posteriors, true_classes):
        # perform backpropagation w.r.t. the prediction for the latest mini-batch X
        # start backpropagation with output layer
        gradient = self.layers[-1].backward(predicted_posteriors, true_classes)

        # propagate gradients backwards through remaining layers
        for layer in reversed(self.layers[:-1]):
            gradient = layer.backward(gradient) # your code here

    def update(self, X, Y, learning_rate):
        posteriors = self.forward(X)
        self.backward(posteriors, Y)
        for layer in self.layers:
            layer.update(learning_rate)

    def train(self, x, y, n_epochs, batch_size, learning_rate):
        N = len(x)
        n_batches = N // batch_size
        for i in range(n_epochs):
            # print("Epoch", i)
            # reorder data for every epoch
            # (i.e. sample mini-batches without replacement)
            permutation = np.random.permutation(N)

            for batch in range(n_batches):
                # create mini-batch
                start = batch * batch_size
                x_batch = x[permutation[start:start+batch_size]]
                y_batch = y[permutation[start:start+batch_size]]

                # perform one forward and backward pass and update network parameters
                self.update(x_batch, y_batch, learning_rate)

# #################################
if __name__ =="__main__":

    # set training / test set size
    N = 2000

    # create training and test data
    X_train, Y_train = datasets.make_moons(N, noise=0.05)
    X_test,  Y_test  = datasets.make_moons(N, noise=0.05)
    n_features = 2
    n_classes  = 2

    # standardize features to be in [-1, 1]
    offset  = X_train.min(axis=0)
    scaling = X_train.max(axis=0) - offset
    X_train = ((X_train - offset) / scaling - 0.5) * 2.0
    X_test  = ((X_test  - offset) / scaling - 0.5) * 2.0

    # set hyperparameters (play with these!)
    # adjusted hyperparameters 
    n_epochs = 250
    batch_size = 50
    learning_rate = 0.03

    networks = [
        [2, 2, n_classes],
        [3, 3, n_classes],
        [5, 5, n_classes],
        [30 , 30, n_classes]
    ]

    for layer_sizes in networks :
        print ("\n network :", layer_sizes)

        # create network
        network = MLP(n_features, layer_sizes)

        # train
        network.train(X_train, Y_train, n_epochs, batch_size, learning_rate)

        # test
        predicted_posteriors = network.forward(X_test)
        # determine class predictions from posteriors by winner-takes-all rule -> choose the class with the highest posterior probability
        predicted_classes = np.argmax(predicted_posteriors, axis =1) # your code here
        # compute and output the error rate of predicted_classes
        error_rate = np.mean(predicted_classes != Y_test) # your code here
        print ("error rate :", error_rate)




 network : [2, 2, 2]
error rate : 0.1115

 network : [3, 3, 2]
error rate : 0.0

 network : [5, 5, 2]
error rate : 0.0

 network : [30, 30, 2]
error rate : 0.0
